<a href="https://colab.research.google.com/github/faziaakhtar-hub/llm-fact-checker/blob/main/llm_fact_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install transformers datasets torch scikit-learn pandas numpy


In [14]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
print("All libraries imported successfully!")

All libraries imported successfully!


In [16]:
# Load the LIAR dataset - a benchmark dataset for fake news detection
dataset = load_dataset("liar")

# Look at the dataset
print(dataset)
print("\nFirst example:")
print(dataset['train'][0])

RuntimeError: Dataset scripts are no longer supported, but found liar.py

In [17]:
# Install the liar dataset manually
!pip install datasets
from datasets import load_dataset

# Try loading with a different approach
dataset = load_dataset("liar", trust_remote_code=True)
print(dataset)
print("\nFirst example:")
print(dataset['train'][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'liar' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'liar' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


RuntimeError: Dataset scripts are no longer supported, but found liar.py

In [18]:
import pandas as pd

# Load LIAR dataset directly from URL
train_url = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/train.tsv"
test_url = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/test.tsv"

columns = ['id', 'label', 'statement', 'subject', 'speaker',
           'job', 'state', 'party', 'barely_true', 'false',
           'half_true', 'mostly_true', 'pants_fire', 'context']

train_df = pd.read_csv(train_url, sep='\t', header=None, names=columns)
test_df = pd.read_csv(test_url, sep='\t', header=None, names=columns)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("\nFirst example:")
print(train_df.head())

Train size: 10240
Test size: 1267

First example:
           id        label                                          statement  \
0   2635.json        false  Says the Annies List political group supports ...   
1  10540.json    half-true  When did the decline of coal start? It started...   
2    324.json  mostly-true  Hillary Clinton agrees with John McCain "by vo...   
3   1123.json        false  Health care reform legislation is likely to ma...   
4   9028.json    half-true  The economic turnaround started at the end of ...   

                              subject         speaker                   job  \
0                            abortion    dwayne-bohac  State representative   
1  energy,history,job-accomplishments  scott-surovell        State delegate   
2                      foreign-policy    barack-obama             President   
3                         health-care    blog-posting                   NaN   
4                        economy,jobs   charlie-crist               

In [19]:
# Data Cleaning
# Simplify labels - convert 6 labels to binary (true/false)
def simplify_label(label):
    if label in ['true', 'mostly-true', 'half-true']:
        return 1  # True
    else:
        return 0  # False (barely-true, false, pants-fire)

train_df['binary_label'] = train_df['label'].apply(simplify_label)
test_df['binary_label'] = test_df['label'].apply(simplify_label)

# Keep only what we need
train_df = train_df[['statement', 'binary_label']].dropna()
test_df = test_df[['statement', 'binary_label']].dropna()

print("Label distribution in training data:")
print(train_df['binary_label'].value_counts())
print("\nSample statements:")
print(train_df.head())

Label distribution in training data:
binary_label
1    5752
0    4488
Name: count, dtype: int64

Sample statements:
                                           statement  binary_label
0  Says the Annies List political group supports ...             0
1  When did the decline of coal start? It started...             1
2  Hillary Clinton agrees with John McCain "by vo...             1
3  Health care reform legislation is likely to ma...             0
4  The economic turnaround started at the end of ...             1


In [20]:
#  Load BERT Tokenizer
from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the statements
def tokenize_data(texts, labels, max_length=128):
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt'
    )
    return encodings, torch.tensor(list(labels))

# Tokenize train and test data
print("Tokenizing training data...")
train_encodings, train_labels = tokenize_data(
    train_df['statement'],
    train_df['binary_label']
)

print("Tokenizing test data...")
test_encodings, test_labels = tokenize_data(
    test_df['statement'],
    test_df['binary_label']
)

print("Done! Shape of train labels:", train_labels.shape)

Tokenizing training data...
Tokenizing test data...
Done! Shape of train labels: torch.Size([10240])


In [22]:
# Create Dataset Class
from torch.utils.data import Dataset

# Create a PyTorch Dataset class
class FactCheckDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# Create dataset objects
train_dataset = FactCheckDataset(train_encodings, train_labels)
test_dataset = FactCheckDataset(test_encodings, test_labels)

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))
print("Dataset created successfully!")

Train dataset size: 10240
Test dataset size: 1267
Dataset created successfully!


In [23]:
# Load BERT Model
from transformers import AutoModelForSequenceClassification

# Load BERT model for binary classification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2  # True or False
)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Move model to device
model = model.to(device)
print("Model loaded successfully!")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda
Model loaded successfully!


In [25]:
# Train the Model
from torch.optim import AdamW
from torch.utils.data import DataLoader

# Use smaller batch and fewer samples to speed up
train_small = torch.utils.data.Subset(train_dataset, range(500))
test_small = torch.utils.data.Subset(test_dataset, range(100))

train_loader = DataLoader(train_small, batch_size=16, shuffle=True)
test_loader = DataLoader(test_small, batch_size=16, shuffle=False)

# Set up optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training loop - only 1 epoch with small data
print("Starting training...")
model.train()

for epoch in range(1):  # just 1 epoch
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

print("Training complete!")

Starting training...
Epoch 1 Loss: 0.6585
Training complete!


In [27]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Evaluate model
print("Evaluating model...")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print results
print("\n=== Model Performance ===")
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 Score: {f1_score(all_labels, all_preds):.4f}")
print("\nDetailed Report:")
print(classification_report(all_labels, all_preds,
      target_names=['False', 'True']))

Evaluating model...

=== Model Performance ===
Accuracy: 0.5100
F1 Score: 0.6202

Detailed Report:
              precision    recall  f1-score   support

       False       0.46      0.23      0.31        47
        True       0.53      0.75      0.62        53

    accuracy                           0.51       100
   macro avg       0.49      0.49      0.47       100
weighted avg       0.49      0.51      0.47       100



In [28]:
def predict_claim(claim):
    # Tokenize the claim
    inputs = tokenizer(
        claim,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()

    label = "✅ TRUE" if prediction == 1 else "❌ FALSE"
    print(f"Claim: {claim}")
    print(f"Prediction: {label}\n")

# Test with some claims
predict_claim("The earth is flat.")
predict_claim("Vaccines cause autism.")
predict_claim("Climate change is caused by human activity.")
predict_claim("The unemployment rate dropped last year.")

Claim: The earth is flat.
Prediction: ❌ FALSE

Claim: Vaccines cause autism.
Prediction: ❌ FALSE

Claim: Climate change is caused by human activity.
Prediction: ✅ TRUE

Claim: The unemployment rate dropped last year.
Prediction: ✅ TRUE

